# Data pipeline

### Objective: Build a reproducible, leakage-free pipeline for daily volatility forecasting that:

* downloads market data,

* cleans and aligns dates across tickers,

* computes returns and forward-looking volatility targets,

* supports walk-forward evaluation and stress testing on 2007–09 (GFC) and 2020 (COVID),

* prevents look-ahead bias by ensuring features at time $t$ only use information for time ≤ $t$.

#### For computing returns/rolling features, I will sort the data by (ticker , date) inside the functions, becasue say shift() must be applied in chronological order within each ticker.

#### The final data returned will be sorted by ( date, ticker) as it helps in time slicing and having time flow top-to-bottom is easier to inspect and debug. It also matches with how panel data is usually viewed: "For each date, here are all assets."

#### The output of every function will be sorted by (date , ticker) and it will be converted into (ticker, date) internally inside a function that uses shift, rolling, or anything order-dependent per ticker before processing it. This way I get correct computation and a dataset that is easy to inspect and slice by time.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import yfinance as yf # Importing yahoo finance to get the stock data

In [33]:
''' Functions defined for sorting'''

def sort_date_ticker(df):
    return df.sort_values(["date", "ticker"]).reset_index(drop=True)

def sort_ticker_date(df):
    return df.sort_values(["ticker", "date"]).reset_index(drop=True)

In [ ]:
# Step 1: Download raw OHLCV data
# OHLCV: Open, High, Low, Close and Volume data



def download_ohlc(tickers, start, end):
    """
    Download daily OHLCV data from Yahoo Finance via yfinance and return a tidy DataFrame.

    Output columns:
      ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']

    Notes:
    - For multiple tickers, yfinance returns MultiIndex columns; we reshape to long/tidy format.
    - We keep auto_adjust=False so we get both raw OHLC and a separate Adj Close column.
    """

    # 1) Download data from yfinance
    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,   # keep raw OHLC; use Adj Close for returns later
        actions=False,       # don't include dividends/splits columns
        group_by="column",   # MultiIndex columns grouped by field then ticker
        progress=False,
        threads=True
    )

    # 2) Convert yfinance output into a tidy/long DataFrame (one row per date per ticker)
    # We do this because it makes multi-assest modeling much simpler and is more general
    # There are multiple rows for a given date (one for each ticker) and the unique key is (date, ticker)
    if isinstance(df.columns, pd.MultiIndex):
        # Multi-ticker case: columns are a MultiIndex (field, ticker)
        tidy = []
        for t in tickers:
            # Extract the columns for one ticker
            sub = df.xs(t, axis=1, level=1).copy()

            # Standardize column names: "Adj Close" -> "adj_close"
            sub.columns = [c.lower().replace(" ", "_") for c in sub.columns]

            # Add ticker identifier and make date a column
            sub["ticker"] = t
            sub = sub.reset_index()  # index -> column (usually named "Date")

            tidy.append(sub)

        out = pd.concat(tidy, ignore_index=True)

    else:
        # Single-ticker case: columns are not MultiIndex
        out = df.copy()
        out.columns = [c.lower().replace(" ", "_") for c in out.columns]
        out = out.reset_index()
        out["ticker"] = tickers[0]

    # 3) Standardize the date column name (yfinance usually uses "Date")
    out = out.rename(columns={"Date": "date", "date": "date"})

    # 4) Standardize adj close naming (defensive)
    out = out.rename(columns={"adjclose": "adj_close"})

    # 5) Validate required columns exist
    needed = {"date", "ticker", "open", "high", "low", "close", "adj_close", "volume"}
    missing = needed - set(out.columns)
    if missing:
        raise ValueError(f"Missing columns from yfinance output: {missing}")

    # 6) Sort for deterministic order (important for later returns/targets)
    out = out.sort_values(["date", "ticker"]).reset_index(drop=True) # First sorting by date and then ticker so that time flows downward

    return out

In [26]:
raw = download_ohlc(['JPM', 'SPY'], start='2000-01-01', end = '2000-02-01')
raw.head(6)

,date,adj_close,close,high,low,open,volume,ticker
0,2000-01-03,22.910496,48.583332,50.250000,48.083332,49.833332,12019200,JPM
1,2000-01-03,91.617058,145.437500,148.250000,143.875000,148.250000,8164300,SPY
2,2000-01-04,22.407806,47.250000,47.458332,46.125000,47.083332,11723400,JPM
3,2000-01-04,88.034286,139.750000,144.062500,139.640625,143.531250,8089800,SPY
4,2000-01-05,22.269482,46.958332,48.375000,46.000000,46.833332,8714550,JPM
5,2000-01-05,88.191788,140.000000,141.531250,137.250000,139.937500,12177900,SPY


In [22]:
# Step 2: Clean and align dates across tickers, drop rows with missing values, remove duplicates, sort by ticker/date, and keep only those dates where all tickers have data

def clean_and_align(raw_data):
    """
    Goal:
    - Remove rows with missing prices
    - Ensure (date, ticker) is unique
    - Keep only dates where ALL tickers have data (common calendar)
    - Return data sorted by ['date', 'ticker'] for readability and consistent downstream use
    """
    # 1) Drop rows where adjusted close is missing (can't compute returns reliably)
    df = raw_data.dropna(subset=["adj_close"]).copy()

    # 2) Sort by date then ticker and remove duplicates
    #    Duplicates can occur rarely due to data issues; we keep one row per (date, ticker).
    df = df.sort_values(["date", "ticker"]).drop_duplicates(subset=["date", "ticker"])

    # 3) Count how many distinct tickers we have in the dataset
    n_tickers = df["ticker"].nunique()

    # 4) For each date, count how many tickers have data on that date
    #    If a date has fewer than n_tickers, at least one ticker is missing that day.
    counts = df.groupby("date")["ticker"].nunique()

    # 5) Keep only dates where we have data for ALL tickers (common dates)
    common_dates = counts[counts == n_tickers].index
    df = df[df["date"].isin(common_dates)].copy()

    # 6) Re-sort after filtering to keep consistent ordering
    df = df.sort_values(["date", "ticker"]).reset_index(drop=True)

    return df

In [23]:
data = clean_and_align(raw)
data.head(6)

,date,adj_close,close,high,low,open,volume,ticker
0,2000-01-03,22.910496,48.583332,50.250000,48.083332,49.833332,12019200,JPM
1,2000-01-03,91.617058,145.437500,148.250000,143.875000,148.250000,8164300,SPY
2,2000-01-04,22.407806,47.250000,47.458332,46.125000,47.083332,11723400,JPM
3,2000-01-04,88.034286,139.750000,144.062500,139.640625,143.531250,8089800,SPY
4,2000-01-05,22.269482,46.958332,48.375000,46.000000,46.833332,8714550,JPM
5,2000-01-05,88.191788,140.000000,141.531250,137.250000,139.937500,12177900,SPY


In [ ]:
# Step 3: Create a stationary series for modeling. Use adjusted close values to take corporate actions into account and compute daily log returns

def add_returns(df):
    """
    Compute daily log returns (percent) from adj_close for each ticker:
        ret_t = 100 * log(adj_close_t / adj_close_{t-1})

    Workflow:
    - Temporarily sort by (ticker, date) to ensure shift() is chronologically correct per ticker
    - Compute returns using groupby().transform() (index-aligned)
    - Drop NaN returns (first row per ticker)
    - Return sorted by (date, ticker)
    """
    out = df.copy()

    # Ensure date is datetime and adj_close is numeric
    out["date"] = pd.to_datetime(out["date"])
    out["adj_close"] = pd.to_numeric(out["adj_close"], errors="coerce")

    # Compute in correct per-ticker chronological order
    tmp = sort_ticker_date(out)
    tmp["ret"] = tmp.groupby("ticker")["adj_close"].transform(
        lambda s: 100.0 * np.log(s / s.shift(1))
    )

    # Drop NaN returns (first observation per ticker + any rows with missing adj_close)
    tmp = tmp.dropna(subset=["ret"]).copy()

    # Return in preferred global order
    return sort_date_ticker(tmp)

In [35]:
data = add_returns(data)
data.head(6)

,date,adj_close,close,high,low,open,volume,ticker,ret
0,2000-01-04,22.407806,47.250000,47.458332,46.125000,47.083332,11723400,JPM,-2.218574
1,2000-01-04,88.034286,139.750000,144.062500,139.640625,143.531250,8089800,SPY,-3.989112
2,2000-01-05,22.269482,46.958332,48.375000,46.000000,46.833332,8714550,JPM,-0.619219
3,2000-01-05,88.191788,140.000000,141.531250,137.250000,139.937500,12177900,SPY,0.178749
4,2000-01-06,22.585651,47.625000,48.625000,46.500000,46.750000,8369250,JPM,1.409761
5,2000-01-06,86.774368,137.750000,141.500000,137.750000,139.625000,6227200,SPY,-1.620257


In [ ]:
# Step 4: Create forward loooking targets while making sure that there is no leakage i.e., avoid using accidentally using future information in featues/training
# Horizons: 1-day and 5-day

def add_targets(df, horizons=(1, 5)):
    """
    Add forward-looking realized variance targets for volatility forecasting.

    Assumes:
    - df contains columns: ['date', 'ticker', 'ret'] where ret is daily log return in percent.

    Targets (variance form):
    - 1-day ahead:  rv1_var(t) = ret_{t+1}^2
    - h-day ahead:  rvh_var(t) = sum_{i=1..h} ret_{t+i}^2

    Key points:
    - These are *future* quantities by design (targets), so we use shift(-1).
    - We compute within each ticker (groupby) and MUST ensure chronological order.
    - We use transform/apply carefully to avoid index misalignment issues.
    - Output is returned sorted by ['date', 'ticker'].
    """
    out = df.copy()

    # Compute per-ticker squared returns (variance proxy at daily frequency)
    out["ret2"] = out["ret"] ** 2

    # Compute targets in correct chronological order per ticker
    tmp = sort_ticker_date(out)

    for h in horizons:
        if h == 1:
            # Next-day realized variance: ret_{t+1}^2 aligned at time t
            tmp[f"rv{h}_var"] = tmp.groupby("ticker")["ret2"].shift(-1)
        else:
            # Next-h-days realized variance:
            # sum of squared returns from t+1 ... t+h, aligned at time t
            tmp[f"rv{h}_var"] = tmp.groupby("ticker")["ret2"].transform(
                lambda s: s.rolling(window=h).sum().shift(-h)
            )

    # Drop rows where any target is NaN (last few rows per ticker won't have future data)
    target_cols = [f"rv{h}_var" for h in horizons]
    tmp = tmp.dropna(subset=target_cols).copy()

    # Return in your preferred global order
    return sort_date_ticker(tmp)

In [39]:
data = add_targets(data)
data.head(6)

,date,adj_close,close,high,low,open,volume,ticker,ret,ret2,rv1_var,rv5_var
0,2000-01-10,22.605398,47.666668,48.916668,47.666668,48.500000,4723500,JPM,-1.733218,3.004046,5.704318,14.393845
1,2000-01-10,92.128868,146.250000,146.906250,145.031250,146.250000,5741700,SPY,0.342426,0.117256,1.449042,36.092660
2,2000-01-11,22.071894,46.541668,46.958332,45.500000,46.666668,8405550,JPM,-2.388371,5.704318,0.390253,14.400665
3,2000-01-11,91.026505,144.500000,146.093750,143.500000,145.812500,7503700,SPY,-1.203761,1.449042,0.999599,37.060308
4,2000-01-12,22.210209,46.833332,47.250000,46.333332,46.458332,7271850,JPM,0.624702,0.390253,2.253484,14.666725
5,2000-01-12,90.120956,143.062500,144.593750,142.875000,144.593750,6907700,SPY,-0.999799,0.999599,1.809606,36.244682


In [40]:
# Step 5: Adding the labels for the regimes of great financial stress e.g., ( 2007-2009 Financial Crisis and 2020 Covid) to be used for slicing and training to make sure
# there is not leakage and we can show performance during crises without contaminating the model training

def add_regime_labels(df, crisis_windows):
    """
    Add a 'regime' label column for later stress-test evaluation (NOT FOR TRAINING).

    Parameters
    ----------
    df : pd.DataFrame
    crisis_windows : dict
        Mapping of regime_name -> (start_date, end_date), e.g.
        {
          "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
          "COVID_2020": ("2020-02-15", "2020-05-31")
        }

    Returns
    -------
    pd.DataFrame
        Same rows as input with an additional 'regime' column.
        Rows outside all crisis windows are labeled 'calm'.

    Notes
    -----
    - This is meant for slicing results (metrics/plots) by regime.
    - Avoid feeding 'regime' into models unless you explicitly design and
      validate a regime-aware approach; otherwise it can create leakage-like issues.
    """
    out = df.copy()

    # Ensure dates are comparable (string comparisons can be fragile)
    out["date"] = pd.to_datetime(out["date"])

    # Default regime: calm (outside the defined stress windows)
    out["regime"] = "calm"

    # Overwrite regime labels for dates that fall inside each crisis window
    for name, (start, end) in crisis_windows.items():
        start_dt = pd.to_datetime(start)
        end_dt = pd.to_datetime(end)

        # Boolean mask: True for rows whose date is inside [start, end]
        mask = (out["date"] >= start_dt) & (out["date"] <= end_dt)
        out.loc[mask, "regime"] = name

    # Keep consistent ordering
    return sort_date_ticker(out)

In [44]:
crisis_windows = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31")
}

data = add_regime_labels(data, crisis_windows)
data.head(10)

,date,adj_close,close,high,low,open,volume,ticker,ret,ret2,rv1_var,rv5_var,regime
0,2000-01-10,22.605398,47.666668,48.916668,47.666668,48.500000,4723500,JPM,-1.733218,3.004046,5.704318,14.393845,calm
1,2000-01-10,92.128868,146.250000,146.906250,145.031250,146.250000,5741700,SPY,0.342426,0.117256,1.449042,36.092660,calm
2,2000-01-11,22.071894,46.541668,46.958332,45.500000,46.666668,8405550,JPM,-2.388371,5.704318,0.390253,14.400665,calm
3,2000-01-11,91.026505,144.500000,146.093750,143.500000,145.812500,7503700,SPY,-1.203761,1.449042,0.999599,37.060308,calm
4,2000-01-12,22.210209,46.833332,47.250000,46.333332,46.458332,7271850,JPM,0.624702,0.390253,2.253484,14.666725,calm
5,2000-01-12,90.120956,143.062500,144.593750,142.875000,144.593750,6907700,SPY,-0.999799,0.999599,1.809606,36.244682,calm
6,2000-01-13,22.546135,47.541668,48.333332,47.041668,47.416668,6918900,JPM,1.501161,2.253484,12.462410,23.814511,calm
7,2000-01-13,91.341469,145.000000,145.750000,143.281250,144.468750,5158300,SPY,1.345216,1.809606,1.818851,6.194354,calm
8,2000-01-14,23.356277,49.250000,50.500000,48.541668,49.291668,9731850,JPM,3.530214,12.462410,15.756480,36.566945,calm
9,2000-01-14,92.581688,146.968750,147.468750,145.968750,146.531250,7437300,SPY,1.348648,1.818851,0.623867,6.700965,calm


In [49]:
# Combining all the above steps together


from pathlib import Path

def make_dataset(tickers, start, end, horizons, crisis_windows=None):
    """
    End-to-end dataset builder for the volatility forecasting project.

    Pipeline objective
    ------------------
    Produce a tidy, leakage-safe panel dataset (one row per date per ticker) with:
      - cleaned + date-aligned OHLCV data
      - daily log returns (percent) computed from adj_close
      - forward-looking realized variance targets (e.g., 1d and 5d)
      - regime labels for stress-test evaluation (GFC/COVID), not for training

    Returns
    -------
    pd.DataFrame
        Sorted by ['date', 'ticker'] with columns including:
        ['date','ticker','open','high','low','close','adj_close','volume','ret','ret2','rv1_var','rv5_var','regime']
        (targets depend on `horizons`)
    """
    # If no crisis window is given then don't label any special regimes and everything stays "calm"
    crisis_windows = {} if crisis_windows is None else crisis_windows

    # 1) Download raw data (tidy format)
    raw = download_ohlc(tickers=tickers, start=start, end=end)

    # 2) Clean & align dates across tickers
    df = clean_and_align(raw)

    # 3) Compute returns (drops NaN returns)
    df = add_returns(df)

    # 4) Create forward-looking targets (drops rows without targets)
    df = add_targets(df, horizons=horizons)

    # 5) Add regime labels (for evaluation slicing)
    df = add_regime_labels(df, crisis_windows=crisis_windows)

    # (Optional) Keep only columns we care about right now
    target_cols = [f"rv{h}_var" for h in horizons]
    keep_cols = [
        "date", "ticker", "open", "high", "low", "close", "adj_close", "volume",
        "ret", "ret2", *target_cols, "regime"
    ]
    df = df[keep_cols].copy()

    # Ensure consistent ordering
    return sort_date_ticker(df)

In [52]:
# Define the scope, start/end dates, and regimes

tickers = ["SPY", "JPM"]
start = "2000-01-01"
end = None  # Gives the latest

horizon = (1,5)
crisis_windows = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31")
}

In [53]:
df = make_dataset(tickers, start, end, horizon, crisis_windows)
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605410,4723500,-1.733118,3.003698,5.704767,14.393754,calm
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128868,5741700,0.342434,0.117261,1.449082,36.092326,calm
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071884,8405550,-2.388465,5.704767,0.390232,14.400553,calm
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026489,7503700,-1.203778,1.449082,0.999633,37.060026,calm
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210196,7271850,0.624685,0.390232,2.253614,14.666814,calm


In [ ]:
# Print few rows from GFC 
df[df["regime"] == "GFC_2007_2009"].head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime
3756,2007-07-02,JPM,48.900002,49.250000,48.570000,49.150002,30.419546,15717900,1.434433,2.057597,1.350090,6.540970,GFC_2007_2009
3757,2007-07-02,SPY,150.869995,151.919998,150.770004,151.789993,107.606117,103357000,0.899987,0.809976,0.130826,2.938345,GFC_2007_2009
3758,2007-07-03,JPM,49.189999,49.580002,49.040001,49.340000,30.775063,7444000,1.161934,1.350090,1.256525,6.085832,GFC_2007_2009
3759,2007-07-03,SPY,152.179993,152.500000,151.990005,152.339996,107.996033,54048400,0.361699,0.130826,0.011040,0.953126,GFC_2007_2009
3760,2007-07-05,JPM,49.099998,49.349998,48.650002,48.790001,30.432016,11613000,-1.120948,1.256525,0.135612,5.896238,GFC_2007_2009


In [ ]:
# Print few rows from covid
df[df["regime"] == "COVID_2020"].head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime
10114,2020-02-18,JPM,137.339996,137.710007,135.050003,135.639999,115.172981,8996400,-1.332868,1.776537,1.835223,3.712392,COVID_2020
10115,2020-02-18,SPY,336.510010,337.670013,335.209991,336.730011,308.567017,57226200,-0.258046,0.066588,0.227555,0.743619,COVID_2020
10116,2020-02-19,JPM,136.910004,138.389999,136.339996,137.490005,116.743851,7071200,1.354704,1.835223,0.000000,3.712392,COVID_2020
10117,2020-02-19,SPY,337.790009,339.079987,337.480011,338.339996,310.042480,48814700,0.477027,0.227555,0.169500,0.500664,COVID_2020
10118,2020-02-20,JPM,137.169998,138.360001,136.529999,137.490005,116.743851,7422800,0.000000,0.000000,1.511534,5.216355,COVID_2020


In [ ]:
# Check if GFC and COVID rows exist and majority are calm
df["regime"].value_counts()

regime
calm             11982
GFC_2007_2009     1008
COVID_2020         144
Name: count, dtype: int64